# SQL Advanced

**Estimated time:** ~14 hours total (about 6 hours reading and running this guide + ~8 hours on
`sql-advanced-exercises.ipynb`).

The intermediate level was about getting the right answer. This one is about getting it in two seconds instead
of two minutes, on a schema that will still make sense in three years, without corrupting anything on the way.

That means four different subjects sharing one notebook: how the engine executes your query and how to make it
execute a better one, the recursive and windowed SQL that the previous level only pointed at, how to design
tables rather than just query them, and the loading patterns that let a report be rebuilt every night without
drifting.

## Who This Is For

You finished `sql-intermediate`, or you write CTEs and window functions without thinking about it. You have
probably had a query that was correct and much too slow, and guessed at an index.

## What You Will Learn

- What the engine does with your query, and how to read `EXPLAIN QUERY PLAN`
- B-tree indexes: single, composite, covering — and why column order decides everything
- Why some conditions cannot use an index at all, and how to rewrite them so they can
- How to time a query honestly, so the "improvement" is real
- Recursive CTEs: hierarchies, date spines, generated series
- Window frames in full — `ROWS` versus `RANGE`, named windows, gaps and islands, sessionization
- Transactions, ACID, isolation levels and what SQLite actually guarantees
- Constraints, generated columns, and pushing correctness into the schema
- Normalization to third normal form, and the deliberate reasons to stop short of it
- Star schemas, slowly changing dimensions, and idempotent incremental loads
- JSON columns for the data that will not sit still
- The rewrites that reliably make things faster, and the anti-patterns that make them slow
- Classic interview problems, solved properly

## How to Use This Guide

Run every cell with **Shift + Enter**, in order. The setup cell builds the familiar shop database **and** a
200,000-row `events` table, because you cannot learn anything about performance from 300 rows.

Timings in this notebook are real and will differ on your machine. What matters is the ratio between two
numbers measured back to back, never the absolute value.

When a section shows a query plan, read it before you read the explanation. `SCAN` means the whole table;
`SEARCH ... USING INDEX` means it jumped straight to the rows. Almost all of query tuning is turning the first
into the second.

**Practice:** `sql-advanced-exercises.ipynb` follows this guide section by section.

## 1. Setup — And a Table Big Enough to Measure

The shop database is the same as before. What is new is `events`: 200,000 rows of synthetic activity, built by
a recursive CTE inside the database rather than loaded from a file.

Two hundred thousand rows is small by production standards and enormous compared to anything in the previous
levels. It is enough that a full scan takes measurable time and an index does not, which is all we need.

Generating it with `WITH RECURSIVE seq(n)` is worth noticing in its own right — it is the standard way to make
a numbers table, and section 10 uses the same trick for dates.

In [1]:
import sqlite3
import pandas as pd

SQL = "assets/sql"          # the bundled schema and CSV files
TABLES = ["categories", "customers", "employees", "products",
          "orders", "order_items", "payments"]

con = sqlite3.connect(":memory:")         # the database lives in RAM -- nothing to clean up

with open(f"{SQL}/schema.sql") as f:
    con.executescript(f.read())           # creates the seven empty tables

con.execute("PRAGMA foreign_keys = ON")   # from here on, SQLite enforces the foreign keys

for table in TABLES:
    pd.read_csv(f"{SQL}/{table}.csv").to_sql(table, con, if_exists="append", index=False)
con.commit()


def q(sql):
    """Run a SELECT and hand the result back as a pandas DataFrame."""
    return pd.read_sql_query(sql, con)


def run(sql):
    """Run statements that change data or structure: CREATE, INSERT, UPDATE, DELETE."""
    con.executescript(sql)
    con.commit()


pd.set_option("display.width", 110)
pd.set_option("display.max_rows", 25)

for table in TABLES:
    print(f"{table:12s} {q(f'SELECT COUNT(*) AS n FROM {table}')['n'][0]:>4} rows")


# ---- a bigger table, so that timings and query plans mean something ----
run("""
DROP TABLE IF EXISTS events;

CREATE TABLE events (
    event_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    event_type  TEXT    NOT NULL,
    event_date  TEXT    NOT NULL,
    amount      REAL    NOT NULL
);

INSERT INTO events (event_id, customer_id, event_type, event_date, amount)
WITH RECURSIVE seq(n) AS (
    SELECT 1
    UNION ALL
    SELECT n + 1 FROM seq WHERE n < 200000
)
SELECT n,
       1 + (n * 7919) % 60,
       CASE WHEN (n * 31) % 100 < 55 THEN 'view'
            WHEN (n * 31) % 100 < 80 THEN 'search'
            WHEN (n * 31) % 100 < 93 THEN 'cart'
            ELSE 'checkout' END,
       date('2023-01-01', '+' || (n % 730) || ' days'),
       ROUND(((n * 37) % 5000) / 10.0, 2)
FROM seq;
""")

print("events      ", q("SELECT COUNT(*) AS n FROM events")["n"][0], "rows")

categories      8 rows
customers      60 rows
employees      15 rows
products       40 rows
orders        300 rows
order_items   673 rows
payments      248 rows
events       200000 rows


**Step by step:**

1. The first part is the usual setup: the seven shop tables built in memory from `../assets/sql/`.
2. `WITH RECURSIVE seq(n) AS (SELECT 1 UNION ALL SELECT n + 1 FROM seq WHERE n < 200000)` counts to 200,000. The
   first `SELECT` is the anchor, the second refers to `seq` itself, and the `WHERE` is what stops it.
3. `INSERT INTO events ... SELECT` writes the generated rows straight into the table. Nothing crosses into
   Python, which is why 200,000 rows land in a fraction of a second.
4. `(n * 7919) % 60` spreads the rows over 60 customers and `(n * 31) % 100` decides the event type, giving a
   realistic mix: about 110,000 views, 50,000 searches, 26,000 carts and 14,000 checkouts. Both multipliers are
   coprime with their modulus, which stops the pattern lining up and clumping.
5. `date('2023-01-01', '+' || (n % 730) || ' days')` spreads them over two years. `||` builds the modifier
   string — `'+417 days'` — which `date()` then applies.

## 2. What the Engine Does With Your Query

SQL is declarative: you describe the result, not the method. Between your text and your rows there are four
stages.

1. **Parse** — is it valid SQL? Do these tables and columns exist?
2. **Rewrite** — the optimizer transforms the query into an equivalent, cheaper one. Views are inlined,
   subqueries are flattened into joins, conditions are pushed down closer to the tables, constant expressions
   are folded.
3. **Plan** — choose the *physical* operations: which table to read first, whether to use an index, how to join
   (nested loop, hash join, merge join), whether a sort is needed. Costs are estimated from **statistics** about
   how many rows there are and how many distinct values a column holds.
4. **Execute** — run the chosen plan.

Two consequences follow, and they are the reason this section exists.

**Your query text is not the execution order.** The optimizer is free to rearrange anything that does not change
the result. Rewriting a query "to make it faster" often changes nothing at all, because the optimizer had
already made that transformation.

**The plan depends on the data, not just the query.** The same query can use an index today and scan tomorrow,
because the statistics changed. `ANALYZE` is what refreshes them, and a plan that regressed overnight is very
often a plan built from stale statistics.

In [2]:
run("CREATE INDEX IF NOT EXISTS idx_events_type ON events(event_type)")
run("ANALYZE")   # collect statistics -- SQLite stores them in sqlite_stat1

print(q("SELECT tbl, idx, stat FROM sqlite_stat1 WHERE tbl IN ('events', 'orders', 'customers') "
        "ORDER BY tbl").to_string(index=False))

print("\nthe optimizer flattens this subquery into a plain join:")
print(q("""
EXPLAIN QUERY PLAN
SELECT o.order_id
FROM orders o
WHERE o.customer_id IN (SELECT customer_id FROM customers WHERE city = 'Chennai')
""").to_string(index=False))

      tbl             idx         stat
customers             NaN           60
   events idx_events_type 200000 50000
   orders             NaN          300

the optimizer flattens this subquery into a plain join:
 id  parent  notused                                                           detail
  2       0       98                                                           SCAN o
  7       0        0                                                  LIST SUBQUERY 1
 20       7       47 SEARCH customers USING AUTOMATIC PARTIAL COVERING INDEX (city=?)
 28       7        0                                              CREATE BLOOM FILTER


**Step by step:**

1. `ANALYZE` walks the tables and records how big they are and how selective each index is, into `sqlite_stat1`.
   PostgreSQL's equivalent runs automatically in the background; SQLite's does not, so on a real SQLite
   database you run it after a big load.
2. The `stat` column holds a row count, and for an index a second number as well: `"200000 50000"` means
   200,000 rows and about 50,000 of them per distinct `event_type`. That second number is what the planner uses
   to decide whether an index is worth using — an index with 50,000 rows per value is not very selective.
3. `EXPLAIN QUERY PLAN` in front of any query prints the plan instead of running it. It is free, and it should
   be the first thing you do to a slow query.
4. The plan shows the `IN (SELECT ...)` as a `LIST SUBQUERY` evaluated **once**, with an automatic index and a
   bloom filter built on the fly to test membership — not as a subquery re-run per order. All of that was
   decided in stages 2 and 3; none of it is in the query you wrote.
5. This is why "I rewrote the subquery as a join and it got faster" is often a coincidence. Check the plan
   before and after; if they are identical, the rewrite did nothing.

## 3. EXPLAIN QUERY PLAN — Reading What Was Decided

SQLite's plan output is one line per operation, and the vocabulary is small:

| You see | It means |
| --- | --- |
| `SCAN t` | every row of `t` is read. Fine for a small table, a problem for a large one |
| `SEARCH t USING INDEX i (col=?)` | the index was used to jump to matching rows |
| `SEARCH t USING COVERING INDEX i` | better still — the index held every column needed, the table was never touched |
| `SEARCH t USING INTEGER PRIMARY KEY (rowid=?)` | the fastest lookup there is |
| `USE TEMP B-TREE FOR ORDER BY` | a sort had to be done at run time, in memory |
| `CORRELATED SCALAR SUBQUERY` | the subquery runs again per row |

The one line to hunt for is `SCAN` on a large table. The one to be suspicious of is `USE TEMP B-TREE`, which
means the engine had no ordered path to your rows and sorted them itself.

Other databases use different words for the same ideas. PostgreSQL's `EXPLAIN` says `Seq Scan`, `Index Scan`
and `Index Only Scan`, and `EXPLAIN ANALYZE` actually runs the query and reports the real row counts next to
the estimates. When those two numbers differ by an order of magnitude, you have found your problem.

In [ ]:
def plan(sql):
    return q("EXPLAIN QUERY PLAN " + sql)["detail"].tolist()

print("full scan          ", plan("SELECT * FROM events WHERE amount > 400"))
print("primary key lookup ", plan("SELECT * FROM events WHERE event_id = 12345"))
print("sort needed        ", plan("SELECT * FROM events ORDER BY amount DESC LIMIT 10"))
print("join               ", plan("""
    SELECT o.order_id, c.name
    FROM orders o JOIN customers c ON c.customer_id = o.customer_id"""))
print("correlated subquery", plan("""
    SELECT p.name, (SELECT COUNT(*) FROM order_items i WHERE i.product_id = p.product_id)
    FROM products p"""))

**Step by step:**

1. The helper pulls just the `detail` column, which is the readable part of the plan. The other columns are
   internal node ids.
2. `WHERE amount > 400` has no index to use, so the plan is `SCAN events` — 200,000 rows read to answer one
   question.
3. `WHERE event_id = 12345` is a primary key lookup. In SQLite an `INTEGER PRIMARY KEY` **is** the row id, so
   this is a direct address, not even an index read.
4. `ORDER BY amount DESC LIMIT 10` adds `USE TEMP B-TREE FOR ORDER BY`: the engine has to sort 200,000 rows to
   find the top ten. An index on `amount` would remove that line entirely, which is the next section.
5. The join plan names the tables in the order the engine chose, which is not necessarily the order you wrote.
   The correlated subquery is announced as such — and being told it runs per row is exactly the warning you
   want.

## 4. Indexes — What a B-Tree Buys You

An index is a sorted copy of one or more columns, plus a pointer back to the row. It is a **B-tree**: a shallow,
wide, balanced tree where finding a value costs a handful of steps regardless of table size. Two hundred
thousand rows is about three levels deep; two billion is about five. That flatness is the whole point.

Without one, answering `WHERE customer_id = 17` means reading every row. With one, it means walking down the
tree and following the pointers.

Indexes are not free:

- Every `INSERT`, `UPDATE` and `DELETE` must update every affected index. A table with eight indexes writes
  nine times.
- They take space, sometimes as much as the table.
- The planner may ignore one it judges unhelpful — reading 60% of a table through an index is *slower* than
  scanning it, because the pointer-following is random access.

Which columns? The ones in `WHERE`, `JOIN ... ON` and `ORDER BY` — and only when they are **selective**, meaning
the value narrows the table down a lot. An index on a boolean flag that is true for half the rows will never be
used. The primary key is indexed for you; foreign keys usually are not, and an unindexed foreign key is the
most common missing index there is.

In [ ]:
import time

def timed(sql, runs=5):
    """Median wall time of running a query, in milliseconds."""
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        q(sql)
        times.append((time.perf_counter() - t0) * 1000)
    return sorted(times)[len(times) // 2]

lookup = "SELECT COUNT(*) AS n FROM events WHERE customer_id = 17"

run("DROP INDEX IF EXISTS idx_events_customer")
before = timed(lookup)
plan_before = q("EXPLAIN QUERY PLAN " + lookup)["detail"][0]

run("CREATE INDEX idx_events_customer ON events(customer_id)")
after = timed(lookup)
plan_after = q("EXPLAIN QUERY PLAN " + lookup)["detail"][0]

print(f"no index : {before:7.2f} ms   {plan_before}")
print(f"index    : {after:7.2f} ms   {plan_after}")
print(f"speedup  : {before / after:.0f}x")

q("SELECT name, sql FROM sqlite_master WHERE type = 'index' AND tbl_name = 'events'")

**Step by step:**

1. `timed` runs the query five times and takes the **median**. A single measurement is noise; the first run of
   anything is unrepresentative because of caching.
2. `DROP INDEX IF EXISTS` first, so the cell measures the same thing however many times you re-run it.
3. Before the index the plan is `SCAN events` and the query reads all 200,000 rows. After it, the plan is
   `SEARCH events USING INDEX`, and it reads a few thousand.
4. The speedup is typically ten to fifty times here, and would be far larger on disk — this database is in
   memory, which is the friendliest possible case for a full scan.
5. `sqlite_master` lists the indexes with the exact `CREATE INDEX` text. That is how you audit what a table
   already has before adding another one.

## 5. Composite Indexes — Column Order Is the Whole Story

An index on `(a, b)` sorts by `a`, and within equal `a` values sorts by `b`. Think of a phone book ordered by
surname then first name.

That gives the **leftmost prefix rule**: an index on `(a, b, c)` can serve queries filtering on `a`, on
`a, b`, or on `a, b, c`. It cannot serve a query filtering only on `b` — there is no way to find all the
Bhavyas without reading every surname.

So the order is a design decision, not a formality:

- **Equality columns first, range columns last.** An index on `(customer_id, event_date)` serves
  `customer_id = 17 AND event_date >= '2024-01-01'` perfectly. Reverse it and the engine can use the date part
  and then must check every row it finds for the customer.
- **A range condition ends the usefulness of the index.** Once one column is matched by `>` or `BETWEEN`,
  columns after it in the index can no longer be used to narrow the search.
- **An index can satisfy `ORDER BY`.** If the index order matches the sort you asked for, the sort disappears
  from the plan.

One well-ordered composite index usually replaces three single-column ones.

In [ ]:
both = ("SELECT COUNT(*) AS n FROM events "
        "WHERE customer_id = 17 AND event_date >= '2024-06-01'")
only_date = "SELECT COUNT(*) AS n FROM events WHERE event_date >= '2024-06-01'"

run("DROP INDEX IF EXISTS idx_events_customer")
run("DROP INDEX IF EXISTS idx_events_cust_date")
run("DROP INDEX IF EXISTS idx_events_date_cust")

run("CREATE INDEX idx_events_cust_date ON events(customer_id, event_date)")
print("index (customer_id, event_date)")
print("  filter on both       :", q("EXPLAIN QUERY PLAN " + both)["detail"][0])
print("  filter on date only  :", q("EXPLAIN QUERY PLAN " + only_date)["detail"][0])

run("DROP INDEX idx_events_cust_date")
run("CREATE INDEX idx_events_date_cust ON events(event_date, customer_id)")
print("\nindex (event_date, customer_id)")
print("  filter on both       :", q("EXPLAIN QUERY PLAN " + both)["detail"][0])
print("  filter on date only  :", q("EXPLAIN QUERY PLAN " + only_date)["detail"][0])

print("\nsorting for free:")
run("DROP INDEX idx_events_date_cust")
run("CREATE INDEX idx_events_cust_date ON events(customer_id, event_date)")
print("  ", q("""EXPLAIN QUERY PLAN
    SELECT * FROM events WHERE customer_id = 17 ORDER BY event_date""")["detail"].tolist())

**Step by step:**

1. With `(customer_id, event_date)`, the two-column filter matches the index exactly: equality on the first
   column, range on the second. That is the best case.
2. The same index cannot **search** on `WHERE event_date >= ...` alone, because the date is the *second*
   column. The plan degrades to `SCAN ... USING COVERING INDEX` — it reads every entry in the index and checks
   each one, which is a full scan wearing a hat. This is the leftmost prefix rule made visible.
3. Flip the index to `(event_date, customer_id)` and the date-only query is served — but the two-column query
   is now worse, because the range on the leading column stops the engine using `customer_id` to narrow
   further.
4. The last plan has no `USE TEMP B-TREE FOR ORDER BY` line. The index already holds this customer's rows in
   date order, so the `ORDER BY` costs nothing. An index that removes a sort is often a bigger win than one
   that removes a scan.
5. Notice that neither index ordering wins both cases. Real tuning is choosing which queries matter, not
   finding an index that helps everything.

## 6. Covering Indexes — Never Touching the Table

Normally an index gets the engine to the right row and it then reads the row from the table for the columns it
still needs. That second step is random I/O and it is often the expensive half.

If the index happens to contain **every column the query touches** — the ones in `SELECT` as well as `WHERE` —
the table is never read at all. That is a **covering index**, and SQLite says so in the plan:
`SEARCH ... USING COVERING INDEX`.

The trick is to add the columns you select to the end of an index you already needed:

```sql
CREATE INDEX idx_events_cust_type_amount ON events(customer_id, event_type, amount);
```

`amount` is not filtered on. It is there so that `SELECT amount` can be answered from the index.

The cost is a wider index — more space, slower writes. Adding three text columns to make one report fast is a
bad trade. Adding one number to make a query on the hot path index-only is usually a good one. PostgreSQL has
`INCLUDE` for exactly this, which stores extra columns in the index without making them part of the sort key.

In [ ]:
report = ("SELECT customer_id, SUM(amount) AS total FROM events "
          "WHERE event_type = 'checkout' GROUP BY customer_id")

run("DROP INDEX IF EXISTS idx_events_type")
run("DROP INDEX IF EXISTS idx_events_type_cust_amount")

run("CREATE INDEX idx_events_type ON events(event_type)")
narrow_time = timed(report)
narrow_plan = q("EXPLAIN QUERY PLAN " + report)["detail"].tolist()

run("DROP INDEX idx_events_type")
run("CREATE INDEX idx_events_type_cust_amount ON events(event_type, customer_id, amount)")
covering_time = timed(report)
covering_plan = q("EXPLAIN QUERY PLAN " + report)["detail"].tolist()

print(f"index(event_type)                        {narrow_time:7.2f} ms")
for line in narrow_plan:
    print("   ", line)
print(f"index(event_type, customer_id, amount)   {covering_time:7.2f} ms")
for line in covering_plan:
    print("   ", line)

**Step by step:**

1. The report filters on `event_type` and then needs `customer_id` and `amount` from every matching row.
2. With only `event_type` indexed, the engine finds the rows through the index and then fetches each one from
   the table for the other two columns. The plan also shows a temporary B-tree for the `GROUP BY`.
3. The wider index holds all three columns, in an order that puts the filter first. The plan changes to
   `USING COVERING INDEX` and the table is not read at all.
4. Because the index is sorted by `event_type` then `customer_id`, the rows arrive already grouped, so the
   temporary B-tree for the `GROUP BY` disappears too. Two wins from one index.
5. The measured gap is smaller here than it would be on disk — in memory, "reading the table" is cheap. On a
   real database with the table on SSD, covering indexes routinely turn seconds into milliseconds.

## 7. When an Index Cannot Help — SARGability

A condition is **SARGable** (search-argument-able) if the engine can use it to narrow an index. The rule is
simple and absolute: **the indexed column must appear on its own, on one side of the comparison.**

The moment you wrap it in a function or arithmetic, the index is useless — the index stores `event_date`, not
`strftime('%Y', event_date)`, and the engine will not evaluate a function for every entry to find out.

| Not SARGable | SARGable rewrite |
| --- | --- |
| `WHERE strftime('%Y', event_date) = '2024'` | `WHERE event_date >= '2024-01-01' AND event_date < '2025-01-01'` |
| `WHERE amount * 2 > 100` | `WHERE amount > 50` |
| `WHERE LOWER(name) = 'aarav'` | store a lowercase column, or index the expression |
| `WHERE name LIKE '%book%'` | nothing helps — a leading `%` cannot use a B-tree |
| `WHERE customer_id + 0 = 17` | `WHERE customer_id = 17` |
| `WHERE CAST(customer_id AS TEXT) = '17'` | fix the type mismatch instead |

The last one is worth dwelling on: comparing a number column to a string forces an implicit conversion, and
implicit conversions are the silent index-killers of production systems.

For the cases where you genuinely want the function, most databases — SQLite, PostgreSQL, and MySQL 8 — let you
index the expression itself: `CREATE INDEX ... ON events(strftime('%Y', event_date))`.

In [ ]:
run("DROP INDEX IF EXISTS idx_events_type_cust_amount")
run("DROP INDEX IF EXISTS idx_events_date")
run("CREATE INDEX idx_events_date ON events(event_date)")

not_sargable = "SELECT COUNT(*) AS n FROM events WHERE strftime('%Y', event_date) = '2024'"
sargable     = ("SELECT COUNT(*) AS n FROM events "
                "WHERE event_date >= '2024-01-01' AND event_date < '2025-01-01'")

print("function on the column:")
print("  ", q("EXPLAIN QUERY PLAN " + not_sargable)["detail"][0], f"  {timed(not_sargable):6.2f} ms")
print("range on the column:")
print("  ", q("EXPLAIN QUERY PLAN " + sargable)["detail"][0], f"  {timed(sargable):6.2f} ms")
print("same answer:", q(not_sargable)["n"][0], "==", q(sargable)["n"][0])

run("CREATE INDEX IF NOT EXISTS idx_events_year ON events(strftime('%Y', event_date))")
print("\nafter indexing the expression itself:")
print("  ", q("EXPLAIN QUERY PLAN " + not_sargable)["detail"][0], f"  {timed(not_sargable):6.2f} ms")

**Step by step:**

1. Both queries return the same count. Only one of them can use the index on `event_date`.
2. `strftime('%Y', event_date) = '2024'` scans, because the index is sorted by date and the engine has no way to
   know which stored dates would produce `'2024'` without computing the function on all of them.
3. The range version is SARGable: the column stands alone, so the engine descends the B-tree to
   `'2024-01-01'` and walks forward until it passes `'2025-01-01'`.
4. Note the half-open range — `>= start AND < next_start`. That pattern is correct for dates, times and
   timestamps alike, and it avoids the `BETWEEN` end-inclusive trap from the basics level.
5. Indexing the expression makes the original query fast without changing it. Useful when the query lives in
   code you cannot edit — but you now maintain two indexes on the same column, so prefer the rewrite.

## 8. Measuring Honestly

Most reported speedups are measurement error. Four rules:

1. **Warm up, then measure the median of several runs.** The first execution pays for cache misses and query
   compilation. One timing is a coin toss.
2. **Change one thing.** Add the index, re-measure, nothing else.
3. **Return the same amount of data.** Comparing a query that returns 10 rows with one that returns 100,000 is
   comparing your network, not your query.
4. **Read the plan as well as the clock.** A query can get faster because the cache warmed up. If the plan did
   not change, neither did the query.

Beware of measuring in your notebook when the real query runs on a server: your timing includes fetching rows
into Python. Wrapping the query in `SELECT COUNT(*) FROM ( ... )` measures the database's work while returning
one row, which is usually what you want to compare.

`EXPLAIN QUERY PLAN` costs nothing and never lies about what was chosen. PostgreSQL's `EXPLAIN (ANALYZE,
BUFFERS)` goes further and reports estimated versus actual rows — a mismatch there is the single most useful
diagnostic in query tuning.

In [ ]:
def benchmark(label, sql, runs=7):
    q(sql)                                    # warm-up, not counted
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        q(sql)
        times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    print(f"{label:34s} median {times[len(times)//2]:7.2f} ms   "
          f"min {times[0]:6.2f}   max {times[-1]:6.2f}")
    return times[len(times) // 2]

wrapped = """
SELECT COUNT(*) AS n FROM (
    SELECT customer_id, SUM(amount) AS total
    FROM events
    WHERE event_type = 'cart'
    GROUP BY customer_id
)
"""

run("DROP INDEX IF EXISTS idx_events_type_cust_amount")
no_index = benchmark("no supporting index", wrapped)
run("CREATE INDEX idx_events_type_cust_amount ON events(event_type, customer_id, amount)")
with_index = benchmark("covering index", wrapped)
print(f"\nspeedup {no_index / with_index:.1f}x")
print("plan:", q("EXPLAIN QUERY PLAN " + wrapped)["detail"].tolist())

**Step by step:**

1. The first `q(sql)` is thrown away. It pays the compilation and cache costs so the measured runs do not.
2. Seven runs, sorted, and the median reported alongside the min and max. A wide gap between min and max means
   the machine is busy and the number should not be trusted.
3. `SELECT COUNT(*) FROM ( ... )` wraps the real query so that only one row crosses back into Python. The
   database still does all the work.
4. The index is created between the two benchmarks and nothing else changes, so the difference is attributable.
5. The plan is printed at the end as corroboration. Faster clock plus changed plan is evidence; faster clock
   with an unchanged plan is noise.

## 9. Recursive CTEs — Walking a Hierarchy

A recursive CTE has two halves joined by `UNION ALL`:

```sql
WITH RECURSIVE tree AS (
    SELECT ... FROM employees WHERE manager_id IS NULL      -- anchor: where to start
    UNION ALL
    SELECT ... FROM employees e JOIN tree t ON t.employee_id = e.manager_id   -- step
)
SELECT * FROM tree
```

The anchor runs once. The recursive half then runs repeatedly, each time joining the table to the rows produced
by the previous round, until a round produces nothing.

This is how you answer "everyone under this manager, however deep" — a question ordinary SQL cannot express,
because you do not know how many joins you would need.

Two things to carry with you. Track the depth by adding `t.depth + 1` in the recursive half; you almost always
want it. And if your data can contain a cycle — A manages B manages A — a recursive CTE will loop forever, so
carry the path as a string and refuse to re-enter it, or cap the depth.

In [1]:
q("""
WITH RECURSIVE org AS (
    SELECT employee_id,
           name,
           role,
           0                       AS depth,
           name                    AS path
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    SELECT e.employee_id,
           e.name,
           e.role,
           o.depth + 1,
           o.path || ' > ' || e.name
    FROM employees e
    JOIN org o ON o.employee_id = e.manager_id
)
SELECT depth,
       SUBSTR('          ', 1, depth * 3) || name AS org_chart,
       role,
       path
FROM org
ORDER BY path
""")

NameError: name 'q' is not defined

**Step by step:**

1. The anchor selects the founder — the one row with no manager — and gives them `depth = 0` and a `path` of
   just their name.
2. The recursive half joins `employees` to `org`, the rows found so far. Round one finds the three managers,
   round two finds their reports, and round three finds nobody, which is what ends it.
3. `o.depth + 1` counts levels. `o.path || ' > ' || e.name` builds the trail from the top, which both reads
   well and guards against cycles: you could add `WHERE o.path NOT LIKE '%' || e.name || '%'`.
4. `SUBSTR('          ', 1, depth * 3) || name` indents by depth, turning the result into a readable org chart.
5. `ORDER BY path` sorts each person directly under their manager, because the path shares its prefix with the
   parent's. Ordering by `depth` instead would list all the managers, then all the reports.

## 10. Recursive CTEs — Date Spines and Generated Series

The second use of recursion has nothing to do with hierarchies: **generating rows that do not exist.**

A monthly report grouped from your data has no row for a month with no sales. The chart then draws a line
straight through the gap, and nobody notices the month is missing rather than zero.

The fix is a **date spine**: generate every month in the range, `LEFT JOIN` the data onto it, and `COALESCE` the
holes to zero.

```sql
WITH RECURSIVE months(m) AS (
    SELECT '2023-01-01'
    UNION ALL
    SELECT date(m, '+1 month') FROM months WHERE m < '2024-12-01'
)
```

The same trick generates numbers (a numbers table), splits a delimited string into rows, and fills in missing
days for a daily active users chart. PostgreSQL has `generate_series()` built in, which is the same idea with
less typing; SQLite does not, so the recursive CTE is how you get it.

In [ ]:
q("""
WITH RECURSIVE months(month_start) AS (
    SELECT '2024-01-01'
    UNION ALL
    SELECT date(month_start, '+1 month')
    FROM months
    WHERE month_start < '2024-12-01'
),
sales AS (
    SELECT strftime('%Y-%m', o.order_date)                   AS month,
           SUM(i.quantity * i.unit_price * (1 - i.discount)) AS revenue,
           COUNT(DISTINCT o.order_id)                        AS orders
    FROM orders o
    JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
      AND o.channel = 'phone'
    GROUP BY month
)
SELECT strftime('%Y-%m', m.month_start)     AS month,
       COALESCE(s.orders, 0)                AS orders,
       ROUND(COALESCE(s.revenue, 0), 2)     AS revenue
FROM months m
LEFT JOIN sales s ON s.month = strftime('%Y-%m', m.month_start)
ORDER BY month
""")

**Step by step:**

1. The `months` CTE starts at January and adds a month at a time until December. Twelve rows, none of which
   come from any table.
2. `date(month_start, '+1 month')` handles month lengths correctly, so there is no arithmetic to get wrong.
3. `sales` is the real data, restricted to the phone channel precisely because it is thin enough to have empty
   months.
4. The `LEFT JOIN` goes **from the spine to the data**, never the other way round. That direction is what
   guarantees twelve rows.
5. `COALESCE(s.revenue, 0)` turns the missing months into honest zeros. Without the spine those months would
   simply be absent, and a chart would join the surrounding points as though nothing had happened.

## 11. Window Frames in Full — ROWS, RANGE and Named Windows

The intermediate level used `ROWS BETWEEN ... `. Here is the rest of it.

- **`ROWS`** counts physical rows. `2 PRECEDING` means the two rows above this one, whatever their values.
- **`RANGE`** counts by value of the `ORDER BY` column. `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`
  includes **every row with the same value as the current row**, not just the ones above it.

That difference matters whenever the sort column has ties. With `RANGE`, tied rows all see the same running
total — the total including all of them. With `ROWS`, each tied row sees a different, partial total. Neither is
wrong; they answer different questions, and the default when you write `ORDER BY` without a frame is `RANGE`,
which is not what most people assume.

**Named windows** stop you repeating yourself. When four columns share a window, define it once:

```sql
SELECT SUM(x) OVER w, AVG(x) OVER w, COUNT(*) OVER w
FROM t
WINDOW w AS (PARTITION BY g ORDER BY d)
```

The `WINDOW` clause sits between `HAVING` and `ORDER BY`. SQLite, PostgreSQL and MySQL 8 all support it, and it
makes a long query dramatically easier to check, because the window definition exists in exactly one place.

In [ ]:
q("""
WITH daily AS (
    SELECT event_date,
           COUNT(*)          AS events,
           ROUND(SUM(amount), 2) AS amount
    FROM events
    WHERE event_date BETWEEN '2024-03-01' AND '2024-03-10'
    GROUP BY event_date
),
tied AS (
    SELECT 'a' AS label, 10 AS score UNION ALL SELECT 'b', 20 UNION ALL
    SELECT 'c', 20       UNION ALL SELECT 'd', 30
)
SELECT label,
       score,
       SUM(score) OVER (ORDER BY score ROWS  BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS by_rows,
       SUM(score) OVER (ORDER BY score RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS by_range
FROM tied
ORDER BY score, label
""")

**Step by step:**

1. The `tied` CTE is four rows with a deliberate tie: two rows scoring 20.
2. `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` is a plain running total. The two tied rows get 30 and
   50 — each one adds itself as it goes past.
3. `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` treats the tie as one position, so **both** rows get 50.
   Same data, same `ORDER BY`, different answer.
4. `ROWS` is what you want for a running balance where every transaction counts separately. `RANGE` is what you
   want when tied rows are genuinely the same point in the sequence — the same day, the same score.
5. Writing `OVER (ORDER BY score)` with no frame gets you the `RANGE` behaviour by default. If you have ever
   had a running total look "one row late" on ties, this was why.

## 12. Gaps and Islands

A family of problems that look unrelated until you see the trick:

- Which dates had no activity? (**gaps**)
- What was the longest streak of consecutive active days? (**islands**)
- Group a user's events into sessions, where a session ends after 30 minutes of silence.
- Collapse rows where a value stayed the same into one row per run.

The classic solution is one line of arithmetic. Number the rows in order with `ROW_NUMBER()`, then subtract that
from the value itself. **Within a consecutive run, the difference is constant** — because both sides increase by
one each step. Group by that difference and each group is one island.

For sessionization the same idea in a different dress: compute the gap to the previous row with `LAG`, mark a
new session when the gap exceeds a threshold, and take a running `SUM` of those marks as the session id.

Both are two window functions and a `GROUP BY`, and both are worth being able to write from memory — this is
one of the most commonly asked SQL interview questions there is.

In [ ]:
q("""
WITH active AS (
    SELECT DISTINCT o.order_date AS d
    FROM orders o
    WHERE o.order_date >= '2024-09-01' AND o.order_date < '2024-12-01'
),
numbered AS (
    SELECT d,
           ROW_NUMBER() OVER (ORDER BY d) AS rn,
           date(d, '-' || ROW_NUMBER() OVER (ORDER BY d) || ' days') AS island_key
    FROM active
),
islands AS (
    SELECT island_key,
           MIN(d)   AS start_date,
           MAX(d)   AS end_date,
           COUNT(*) AS days
    FROM numbered
    GROUP BY island_key
)
SELECT start_date,
       end_date,
       days
FROM islands
WHERE days >= 2
ORDER BY days DESC, start_date
""")

**Step by step:**

1. `active` is the list of distinct dates on which any order was placed — the raw sequence with holes in it.
2. `numbered` gives each date its position in the sequence, then subtracts that many days from the date itself.
   Consecutive dates all produce the **same** `island_key`, because each moves forward one day and back one
   position.
3. A gap breaks the pattern: skip a day and every later date gets a new key. So each distinct key is exactly one
   unbroken run.
4. `islands` collapses each key to its first date, last date and length. `MIN` and `MAX` over a group of
   consecutive dates are the run's boundaries.
5. `WHERE days >= 2` keeps the actual streaks. To find the **gaps** instead, `LAG` the end dates and report
   where the next start is more than a day later — the same shape, one function different.

## 13. Transactions, ACID and Isolation

A transaction is a group of statements that either all take effect or none do. The guarantees have a name:

- **Atomic** — all or nothing. A crash in the middle leaves the database as if you had never started.
- **Consistent** — constraints hold at the end. You cannot commit a state that violates a foreign key.
- **Isolated** — concurrent transactions do not see each other's unfinished work.
- **Durable** — once `COMMIT` returns, the data survives a power cut.

Isolation is the one with dials. The standard defines four levels, trading correctness for concurrency:

| Level | Allows |
| --- | --- |
| Read uncommitted | dirty reads — you see another transaction's uncommitted changes |
| Read committed | non-repeatable reads — the same query twice gives different answers |
| Repeatable read | phantoms — new rows appear in a range you already read |
| Serializable | nothing; the result is as if transactions ran one after another |

PostgreSQL defaults to read committed. SQLite is effectively **serializable**, because it takes a database-wide
write lock — simple and correct, but it means one writer at a time. `PRAGMA journal_mode = WAL` improves this
considerably: readers no longer block the writer, and the writer no longer blocks readers.

The practical rules: keep transactions **short**, do not hold one open while waiting on a human or a network
call, and always touch multiple tables in the **same order** everywhere in your code — the classic deadlock is
two transactions grabbing the same two tables in opposite orders.

In [ ]:
import os
import tempfile

demo = os.path.join(tempfile.gettempdir(), "sql_advanced_txn_demo.db")
if os.path.exists(demo):
    os.remove(demo)

w = sqlite3.connect(demo)
w.executescript("""
    CREATE TABLE account (id INTEGER PRIMARY KEY, owner TEXT, balance REAL NOT NULL CHECK (balance >= 0));
    INSERT INTO account (id, owner, balance) VALUES (1, 'Asha', 5000), (2, 'Bala', 5000);
""")
w.commit()
print("journal mode:", w.execute("PRAGMA journal_mode = WAL").fetchone()[0])

r = sqlite3.connect(demo)          # a second connection, standing in for another user

def balances(conn, label):
    rows = conn.execute("SELECT owner, balance FROM account ORDER BY id").fetchall()
    print(f"{label:28s}", dict(rows))

# a transfer that works
w.execute("UPDATE account SET balance = balance - 1500 WHERE id = 1")
w.execute("UPDATE account SET balance = balance + 1500 WHERE id = 2")
balances(r, "other connection, mid-txn")     # still sees the old state
w.commit()
balances(r, "other connection, after commit")

# a transfer that must not happen: the CHECK constraint refuses to let Asha go negative
try:
    w.execute("UPDATE account SET balance = balance - 99999 WHERE id = 1")
    w.execute("UPDATE account SET balance = balance + 99999 WHERE id = 2")
    w.commit()
except Exception as exc:
    w.rollback()
    print("rejected and rolled back:", exc)

balances(w, "final")
print("total unchanged:", w.execute("SELECT SUM(balance) FROM account").fetchone()[0])
w.close(); r.close(); os.remove(demo)

**Step by step:**

1. This one section uses a real file rather than the in-memory database, because two connections cannot share
   an in-memory one — and two connections are the entire point.
2. `PRAGMA journal_mode = WAL` switches to write-ahead logging. Readers now see a consistent snapshot while a
   writer is working, instead of being locked out.
3. The two `UPDATE`s run on `w` without a commit. The reader `r` still reports the old balances — that is
   isolation: unfinished work is invisible.
4. After `w.commit()` the reader sees the new state. The transfer was atomic; there was no instant at which
   1500 existed in neither account or both.
5. The second transfer would push Asha below zero, and the `CHECK (balance >= 0)` constraint refuses it. The
   `rollback()` undoes the whole attempt, and the final total is unchanged — which is the invariant a transfer
   must never break.

## 14. Constraints and Generated Columns — Correctness in the Schema

Every rule you enforce in application code is a rule somebody can bypass with a script, a migration or a second
service. Every rule in the schema is enforced for everyone, forever.

| Constraint | Stops |
| --- | --- |
| `NOT NULL` | missing values |
| `PRIMARY KEY` | duplicates and `NULL` in the identifier |
| `UNIQUE (a, b)` | duplicate combinations — a customer favouriting the same product twice |
| `REFERENCES t(c)` | orphans pointing at rows that do not exist |
| `CHECK (expr)` | anything you can write as a condition — negative prices, statuses off the list |
| `DEFAULT` | not a constraint, but stops "we forgot to set it" |

`ON DELETE` decides what happens to children when a parent is removed: `CASCADE` deletes them,
`SET NULL` orphans them deliberately, `RESTRICT` refuses the delete. `CASCADE` is convenient and occasionally
alarming — deleting one customer can quietly remove years of orders — so use it where the child genuinely
cannot exist alone.

A **generated column** is computed from the others and cannot be set by hand. `VIRTUAL` computes on read and
costs nothing to store; `STORED` computes on write and can be indexed. It is the honest way to keep a derived
value — a line total, a lowercase search key — that can never drift out of step with what it is derived from.

In [ ]:
run("""
DROP TABLE IF EXISTS invoice_line;

CREATE TABLE invoice_line (
    line_id     INTEGER PRIMARY KEY,
    order_id    INTEGER NOT NULL REFERENCES orders(order_id) ON DELETE CASCADE,
    product_id  INTEGER NOT NULL REFERENCES products(product_id),
    quantity    INTEGER NOT NULL CHECK (quantity > 0),
    unit_price  REAL    NOT NULL CHECK (unit_price >= 0),
    discount    REAL    NOT NULL DEFAULT 0 CHECK (discount BETWEEN 0 AND 0.5),
    line_total  REAL    GENERATED ALWAYS AS (quantity * unit_price * (1 - discount)) STORED,
    UNIQUE (order_id, product_id)
);

INSERT INTO invoice_line (order_id, product_id, quantity, unit_price, discount) VALUES
    (1, 15, 2, 4500, 0.05),
    (1, 17, 1,  950, 0.00),
    (2, 15, 3, 4500, 0.10);
""")

for label, sql in [
    ("quantity of zero",        "INSERT INTO invoice_line (order_id, product_id, quantity, unit_price) VALUES (3, 1, 0, 100)"),
    ("60% discount",            "INSERT INTO invoice_line (order_id, product_id, quantity, unit_price, discount) VALUES (3, 2, 1, 100, 0.6)"),
    ("same product twice",      "INSERT INTO invoice_line (order_id, product_id, quantity, unit_price) VALUES (1, 15, 1, 4500)"),
    ("order that does not exist", "INSERT INTO invoice_line (order_id, product_id, quantity, unit_price) VALUES (99999, 1, 1, 100)"),
    ("writing to a generated column", "INSERT INTO invoice_line (order_id, product_id, quantity, unit_price, line_total) VALUES (3, 3, 1, 100, 7)"),
]:
    try:
        run(sql)
        print(f"{label:32s} ACCEPTED -- the schema is not protecting you")
    except Exception as exc:
        print(f"{label:32s} rejected: {str(exc).splitlines()[0]}")

q("SELECT * FROM invoice_line ORDER BY line_id")

**Step by step:**

1. Five different constraints on one small table, each one a class of bad row that can no longer be written.
2. The loop tries to break each of them in turn and prints the rejection. Every attempt fails, which is the
   demonstration: the rules hold regardless of which program is writing.
3. `UNIQUE (order_id, product_id)` is a table-level constraint over a **pair** of columns. Either column may
   repeat; the combination may not.
4. `line_total` is `GENERATED ALWAYS AS (...) STORED`, so it appears in the result without ever being inserted,
   and the attempt to write to it directly is refused. It cannot disagree with the columns it is computed from.
5. `ON DELETE CASCADE` on `order_id` means deleting an order takes its invoice lines with it. Convenient, and
   worth writing down somewhere visible, because the person who runs that `DELETE` in two years will not know.

## 15. Normalization — And When to Stop

Normalization is the process of removing duplicated facts from a schema. The three forms that matter:

- **1NF** — one value per cell. No comma-separated lists in a column, no `phone1`, `phone2`, `phone3`.
- **2NF** — 1NF, and every non-key column depends on the **whole** primary key. In a table keyed by
  `(order_id, product_id)`, a `customer_city` column breaks this: it depends on the order alone.
- **3NF** — 2NF, and no non-key column depends on another non-key column. Storing `city` and `state` together
  breaks it, because state follows from city.

The reason is not tidiness. A duplicated fact can disagree with itself. If a customer's city is stored on 40
order rows, one update that misses a row leaves the database holding two different truths and no way to tell
which is right.

**When to stop.** Normalization optimises for writes and integrity; reading a fully normalized schema means
joining a lot of tables. Analytics workloads deliberately go the other way, and that is section 16. The
sequence that works is: normalize by default, and denormalize afterwards, on purpose, for a measured reason,
with something that keeps the copy in step.

In [ ]:
run("""
DROP TABLE IF EXISTS orders_flat;
CREATE TABLE orders_flat (
    order_id      INTEGER PRIMARY KEY,
    customer_name TEXT,
    customer_city TEXT,
    customer_state TEXT,
    products      TEXT       -- a comma-separated list: not even in first normal form
);

INSERT INTO orders_flat
SELECT o.order_id,
       c.name,
       c.city,
       c.state,
       GROUP_CONCAT(p.name, ', ')
FROM orders o
JOIN customers c   ON c.customer_id = o.customer_id
JOIN order_items i ON i.order_id = o.order_id
JOIN products p    ON p.product_id = i.product_id
GROUP BY o.order_id, c.name, c.city, c.state;
""")

print(q("SELECT * FROM orders_flat ORDER BY order_id LIMIT 3").to_string(index=False))

print("\nthe same city is now stored", q("""
    SELECT COUNT(*) AS n FROM orders_flat WHERE customer_city = 'Chennai'
""")["n"][0], "times instead of once")

print("\n'which orders contain a Vault 1TB SSD?' -- impossible to index, and wrong on substrings:")
print(q("""
SELECT COUNT(*) AS matched
FROM orders_flat
WHERE products LIKE '%Vault 1TB SSD%'
""").to_string(index=False))

print("\nagainst the normalized tables it is an ordinary join:")
print(q("""
SELECT COUNT(DISTINCT i.order_id) AS matched
FROM order_items i
JOIN products p ON p.product_id = i.product_id
WHERE p.name = 'Vault 1TB SSD'
""").to_string(index=False))

**Step by step:**

1. `orders_flat` is what a spreadsheet exported to a database looks like: everything on one row, products in a
   comma-separated string.
2. It breaks 1NF immediately. `products` holds many values in one cell, so you cannot join on it, index it, or
   count it without string surgery.
3. It breaks 3NF too: `customer_state` depends on `customer_city`, which depends on the customer, not on the
   order. Chennai's state is now recorded on every Chennai order, and one bad update makes them disagree.
4. Searching it needs `LIKE '%...%'`, which cannot use an index and matches substrings by accident — a product
   called `Vault 1TB SSD Pro` would be counted as a match.
5. The normalized version answers the same question with a join on a key. Faster, exact, and it stays correct
   when a product is renamed.

## 16. Star Schemas — Facts and Dimensions

Analytical databases denormalize on purpose, in a specific shape.

- A **fact table** holds the measurements — one row per event, at the finest grain you will ever need. It is
  long, narrow, and made of foreign keys plus numbers. `fact_sales` with one row per order line.
- **Dimension tables** hold the descriptive attributes you group and filter by: `dim_product`, `dim_customer`,
  `dim_date`. Short, wide, and denormalized on purpose — `dim_product` holds the category name directly rather
  than pointing at a category table.

Draw it and it looks like a star: the fact table in the middle, dimensions around the outside, every query one
join deep.

Why it wins for analytics: every question is "sum this measure, sliced by those attributes", and that is one
join from the fact to each dimension it needs, with no chains. The measures live in one place, so two teams
cannot compute revenue differently.

A **date dimension** is the piece people skip and then regret. A table with one row per day carrying the
quarter, the week, the day name, the holiday flag and the fiscal period turns "revenue by fiscal quarter,
weekdays only" into a join, instead of a pile of `strftime` calls that each team writes slightly differently.

In [ ]:
run("""
DROP TABLE IF EXISTS fact_sales;
DROP TABLE IF EXISTS dim_product;
DROP TABLE IF EXISTS dim_customer;
DROP TABLE IF EXISTS dim_date;

CREATE TABLE dim_product (
    product_key INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category     TEXT NOT NULL,       -- denormalized on purpose
    list_price   REAL NOT NULL
);
CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    city          TEXT,
    state         TEXT
);
CREATE TABLE dim_date (
    date_key   TEXT PRIMARY KEY,      -- 'YYYY-MM-DD'
    year       INTEGER NOT NULL,
    quarter    TEXT    NOT NULL,
    month      TEXT    NOT NULL,
    day_name   TEXT    NOT NULL,
    is_weekend INTEGER NOT NULL
);
CREATE TABLE fact_sales (
    sale_id      INTEGER PRIMARY KEY,
    date_key     TEXT    NOT NULL REFERENCES dim_date(date_key),
    customer_key INTEGER NOT NULL REFERENCES dim_customer(customer_key),
    product_key  INTEGER NOT NULL REFERENCES dim_product(product_key),
    quantity     INTEGER NOT NULL,
    revenue      REAL    NOT NULL
);

INSERT INTO dim_product SELECT p.product_id, p.name, c.name, p.price
    FROM products p JOIN categories c ON c.category_id = p.category_id;

INSERT INTO dim_customer SELECT customer_id, name, city, state FROM customers;

INSERT INTO dim_date
WITH RECURSIVE d(day) AS (
    SELECT '2023-01-01' UNION ALL SELECT date(day, '+1 day') FROM d WHERE day < '2024-12-31'
)
SELECT day,
       CAST(strftime('%Y', day) AS INTEGER),
       strftime('%Y', day) || '-Q' || ((CAST(strftime('%m', day) AS INTEGER) + 2) / 3),
       strftime('%Y-%m', day),
       CASE strftime('%w', day) WHEN '0' THEN 'Sun' WHEN '1' THEN 'Mon' WHEN '2' THEN 'Tue'
            WHEN '3' THEN 'Wed' WHEN '4' THEN 'Thu' WHEN '5' THEN 'Fri' ELSE 'Sat' END,
       CASE WHEN strftime('%w', day) IN ('0', '6') THEN 1 ELSE 0 END
FROM d;

INSERT INTO fact_sales (date_key, customer_key, product_key, quantity, revenue)
SELECT o.order_date, o.customer_id, i.product_id, i.quantity,
       ROUND(i.quantity * i.unit_price * (1 - i.discount), 2)
FROM orders o
JOIN order_items i ON i.order_id = o.order_id
WHERE o.status <> 'cancelled';
""")

print("fact_sales", q("SELECT COUNT(*) AS n FROM fact_sales")["n"][0], "rows |",
      "dim_date", q("SELECT COUNT(*) AS n FROM dim_date")["n"][0], "rows")

q("""
SELECT d.quarter,
       p.category,
       ROUND(SUM(f.revenue), 2) AS revenue,
       SUM(f.quantity)          AS units
FROM fact_sales f
JOIN dim_date    d ON d.date_key    = f.date_key
JOIN dim_product p ON p.product_key = f.product_key
WHERE d.is_weekend = 0
  AND d.year = 2024
GROUP BY d.quarter, p.category
ORDER BY d.quarter, revenue DESC
LIMIT 12
""")

**Step by step:**

1. `dim_product` stores the category **name**, not a `category_id`. That duplication is the deliberate
   denormalization: a dimension is written rarely and read constantly.
2. `dim_date` is generated by a recursive CTE — 730 rows, one per day, carrying the derived attributes that
   would otherwise be computed in every query.
3. `fact_sales` is one row per order line, holding only keys and two measures. Cancelled orders are filtered
   out here, once, so no downstream query can forget.
4. The report joins the fact to two dimensions and groups. `WHERE d.is_weekend = 0` is a plain column
   comparison instead of a `strftime` call — SARGable, indexable, and the same for everybody.
5. Adding "revenue by day of week" or "revenue by fiscal quarter" needs no new SQL logic, only another column
   in `dim_date`. That extensibility is what the shape buys.

## 17. Slowly Changing Dimensions

A customer moves from Chennai to Mumbai. What should last year's report say?

- **Type 1 — overwrite.** Update the city. Simple, and it rewrites history: last year's Chennai revenue is now
  Mumbai revenue.
- **Type 2 — new row.** Close the old row with an end date, insert a new one, and keep both. Each fact points at
  the version that was current when it happened, so history is preserved.

Type 2 is the standard for anything you report on over time. The shape is always:

| customer_key | customer_id | city | valid_from | valid_to | is_current |
| --- | --- | --- | --- | --- | --- |
| 1 | 42 | Chennai | 2023-01-01 | 2024-05-31 | 0 |
| 2 | 42 | Mumbai | 2024-06-01 | 9999-12-31 | 1 |

Two things make it work. The **surrogate key** (`customer_key`) is unique per *version*, while the natural key
(`customer_id`) identifies the person — so the fact table pointing at `customer_key` is automatically pointing
at the right version. And `valid_to` uses a far-future sentinel rather than `NULL`, so a `BETWEEN` finds the
current row without a special case.

The cost is that "the customer's current city" is now `WHERE is_current = 1` rather than a plain lookup. That
flag exists precisely to make the common case cheap.

In [ ]:
run("""
DROP TABLE IF EXISTS dim_customer_scd;

CREATE TABLE dim_customer_scd (
    customer_key INTEGER PRIMARY KEY,
    customer_id  INTEGER NOT NULL,
    name         TEXT    NOT NULL,
    city         TEXT,
    valid_from   TEXT    NOT NULL,
    valid_to     TEXT    NOT NULL DEFAULT '9999-12-31',
    is_current   INTEGER NOT NULL DEFAULT 1
);

INSERT INTO dim_customer_scd (customer_id, name, city, valid_from)
SELECT customer_id, name, city, '2023-01-01' FROM customers WHERE customer_id <= 5;
""")

print("before the move:")
print(q("SELECT customer_key, customer_id, name, city, valid_from, valid_to, is_current "
        "FROM dim_customer_scd WHERE customer_id = 3").to_string(index=False))

# customer 3 moves on 2024-06-01 -- close the old version, open a new one
run("""
UPDATE dim_customer_scd
   SET valid_to = '2024-05-31', is_current = 0
 WHERE customer_id = 3 AND is_current = 1;

INSERT INTO dim_customer_scd (customer_id, name, city, valid_from)
SELECT customer_id, name, 'Mumbai', '2024-06-01'
FROM dim_customer_scd
WHERE customer_id = 3 AND is_current = 0;
""")

print("\nafter the move:")
print(q("SELECT customer_key, customer_id, name, city, valid_from, valid_to, is_current "
        "FROM dim_customer_scd WHERE customer_id = 3 ORDER BY valid_from").to_string(index=False))

print("\nwhere did customer 3 live when each order was placed?")
q("""
SELECT o.order_id,
       o.order_date,
       d.city AS city_at_the_time
FROM orders o
JOIN dim_customer_scd d
  ON d.customer_id = o.customer_id
 AND o.order_date BETWEEN d.valid_from AND d.valid_to
WHERE o.customer_id = 3
ORDER BY o.order_date
LIMIT 8
""")

**Step by step:**

1. The dimension starts with one row per customer, valid from the beginning of time as far as this warehouse is
   concerned, and `valid_to` defaulting to the far-future sentinel.
2. The move is two statements, and they belong in one transaction: close the old version by setting `valid_to`
   and `is_current = 0`, then insert the new version starting the next day.
3. The date ranges must not overlap and must not have holes. `valid_to = '2024-05-31'` and
   `valid_from = '2024-06-01'` is contiguous — an off-by-one here either double-counts or loses a day.
4. The last query joins on `o.order_date BETWEEN d.valid_from AND d.valid_to`, so each order finds the version
   of the customer that was current on the day. Orders before June say Chennai; orders after say Mumbai.
5. That is the payoff. A type 1 overwrite would have made every one of those orders say Mumbai, and last year's
   regional report would silently change every time somebody moved house.

## 18. Idempotent Incremental Loads

A job that runs every night has to survive being run twice — because it will be. A rerun after a failure, a
backfill, a retry, someone testing. **Idempotent** means running it again changes nothing.

Three patterns, in ascending order of usefulness:

1. **Delete then insert, by partition.** `DELETE FROM fact WHERE date_key = '2024-06-01'` then insert that day.
   Simple, obviously correct, and the standard for daily batches. Keep both statements in one transaction.
2. **Upsert.** `INSERT ... ON CONFLICT (key) DO UPDATE`. Right when rows arrive one at a time and can be
   revised, and it needs a real unique key to conflict on.
3. **Merge on a watermark.** Keep the highest `updated_at` you have processed, select everything above it, and
   move the watermark. This is how incremental loads avoid rescanning history — and the reason it must be
   `>=` with deduplication rather than `>` is that two rows can share a timestamp.

What makes a load non-idempotent, almost always: a plain `INSERT` with no key to conflict on, so a rerun
appends the same rows again. The row count doubles, every number is twice what it should be, and nothing
errors.

In [ ]:
run("""
DROP TABLE IF EXISTS daily_revenue;
CREATE TABLE daily_revenue (
    date_key TEXT PRIMARY KEY,
    orders   INTEGER NOT NULL,
    revenue  REAL    NOT NULL,
    loaded_at TEXT   NOT NULL
);
""")

def load_day(day):
    """Load one day, idempotently: delete the partition, then insert it, in one transaction."""
    con.execute("BEGIN")
    con.execute("DELETE FROM daily_revenue WHERE date_key = ?", (day,))
    con.execute("""
        INSERT INTO daily_revenue (date_key, orders, revenue, loaded_at)
        SELECT o.order_date,
               COUNT(DISTINCT o.order_id),
               ROUND(SUM(i.quantity * i.unit_price * (1 - i.discount)), 2),
               'run-1'
        FROM orders o
        JOIN order_items i ON i.order_id = o.order_id
        WHERE o.order_date = ? AND o.status <> 'cancelled'
        GROUP BY o.order_date
    """, (day,))
    con.commit()

for day in ["2024-11-05", "2024-11-11", "2024-11-19"]:
    load_day(day)
print("after the first run: ", q("SELECT COUNT(*) AS rows_, ROUND(SUM(revenue), 2) AS revenue "
                                 "FROM daily_revenue").to_dict("records")[0])

for day in ["2024-11-05", "2024-11-11", "2024-11-19"]:
    load_day(day)                      # exactly the same job, run again
print("after running it again:", q("SELECT COUNT(*) AS rows_, ROUND(SUM(revenue), 2) AS revenue "
                                    "FROM daily_revenue").to_dict("records")[0])

# what a naive load would have done
con.execute("""INSERT INTO daily_revenue (date_key, orders, revenue, loaded_at)
               SELECT date_key || '-dup', orders, revenue, 'naive-rerun' FROM daily_revenue""")
con.commit()
print("after a load with no key: ", q("SELECT COUNT(*) AS rows_, ROUND(SUM(revenue), 2) AS revenue "
                                       "FROM daily_revenue").to_dict("records")[0],
      " <- doubled, silently")

q("SELECT * FROM daily_revenue WHERE loaded_at = 'run-1' ORDER BY date_key")

**Step by step:**

1. `load_day` is the delete-then-insert pattern. The `DELETE` removes whatever that day already had, so the
   `INSERT` cannot duplicate it.
2. Both statements sit inside `BEGIN ... commit()`. Without the transaction, a crash between them leaves the day
   deleted and not reloaded — a hole in the data that nothing would report.
3. The parameters are passed with `?` placeholders rather than string formatting. That is how you avoid SQL
   injection, and it lets the engine reuse the compiled statement.
4. Running the same three days again produces identical numbers. That is the property worth having: reruns are
   boring.
5. The last block simulates the naive alternative — an `INSERT` with nothing to conflict on. The row count and
   the revenue both double, with no error, which is exactly how a doubled dashboard happens.

## 19. JSON — For the Data That Will Not Sit Still

Sometimes a column genuinely holds a document: an API response, an event payload, a settings blob whose shape
differs per row. Modelling that as 40 mostly-null columns is worse than storing the JSON.

SQLite's JSON functions (built in since 3.38), with PostgreSQL's `jsonb` operators alongside:

| Task | SQLite | PostgreSQL |
| --- | --- | --- |
| Pull a field | `json_extract(doc, '$.city')` | `doc->>'city'` |
| Nested field | `json_extract(doc, '$.address.city')` | `doc#>>'{address,city}'` |
| Array element | `json_extract(doc, '$.tags[0]')` | `doc->'tags'->>0` |
| Explode an array into rows | `json_each(doc, '$.tags')` | `jsonb_array_elements(doc->'tags')` |
| Is it valid? | `json_valid(doc)` | the `jsonb` type rejects it on write |

Two warnings. JSON columns cannot be indexed as such — you index the **extracted expression**, or a generated
column holding it. And the flexibility is a loan: nothing stops one row spelling it `city` and another `City`,
and no constraint will catch it. Use JSON for what is genuinely variable, promote to real columns anything you
filter or join on.

In [ ]:
run("""
DROP TABLE IF EXISTS event_log;

CREATE TABLE event_log (
    log_id  INTEGER PRIMARY KEY,
    payload TEXT NOT NULL CHECK (json_valid(payload)),
    device  TEXT GENERATED ALWAYS AS (json_extract(payload, '$.device')) STORED
);

INSERT INTO event_log (payload) VALUES
    ('{"customer": 3, "device": "mobile", "action": "checkout", "cart": {"items": 2, "value": 5400},
       "tags": ["promo", "returning"]}'),
    ('{"customer": 7, "device": "desktop", "action": "search", "query": "ssd",
       "tags": ["organic"]}'),
    ('{"customer": 3, "device": "mobile", "action": "view", "cart": {"items": 0, "value": 0},
       "tags": ["promo", "app-push", "returning"]}');

CREATE INDEX idx_event_log_device ON event_log(device);
""")

print(q("""
SELECT log_id,
       json_extract(payload, '$.customer')     AS customer_id,
       device,
       json_extract(payload, '$.action')       AS action,
       json_extract(payload, '$.cart.value')   AS cart_value
FROM event_log
ORDER BY log_id
""").to_string(index=False))

print("\ntags exploded into rows:")
print(q("""
SELECT e.log_id, t.value AS tag
FROM event_log e, json_each(e.payload, '$.tags') t
ORDER BY e.log_id, tag
""").to_string(index=False))

print("\nthe generated column is indexed, so this is a SEARCH, not a SCAN:")
print(q("EXPLAIN QUERY PLAN SELECT * FROM event_log WHERE device = 'mobile'")["detail"][0])

**Step by step:**

1. `CHECK (json_valid(payload))` refuses malformed JSON at the door. Without it the column is a text field that
   happens to usually contain JSON.
2. `json_extract(payload, '$.customer')` pulls a top-level field; `'$.cart.value'` walks into a nested object.
   The `$` is the document root.
3. Row 2 has no `cart`, so its `cart_value` is `NULL` rather than an error. That tolerance is the whole appeal
   of JSON and also its risk — a typo'd field name is indistinguishable from a missing one.
4. `json_each(payload, '$.tags')` turns an array into rows, one per element. The comma in the `FROM` is an
   implicit join to that table-valued function, which is the idiom for exploding an array.
5. `device` is a `STORED` generated column over `json_extract`, so it can be indexed — and the plan confirms a
   `SEARCH`. That is the pattern: keep the document, promote the fields you query.

## 20. Rewrites That Actually Make a Difference

Most "optimizations" do nothing, because the optimizer already applied them. These four are real, because each
changes how much work there is rather than how it is spelled.

**1. Correlated subquery in the `SELECT` list → one join or one window function.** A correlated subquery runs
per row. Aggregating the inner table once and joining is one pass instead of N.

**2. `OR` across different columns → `UNION ALL` of two indexed halves.** A single index cannot satisfy
`WHERE a = 1 OR b = 2`, so the engine scans. Two queries, each hitting its own index, stacked with `UNION ALL`,
often beat it — at the cost of deduplicating if the halves can overlap.

**3. N+1 queries → one query.** A loop in your application issuing one query per row is the most expensive
pattern in this list, and the one least visible in the database's logs. Every round trip has latency; a hundred
of them dwarf the work.

**4. `DISTINCT` masking a fan-out → fix the join.** `SELECT DISTINCT` over a join that produced duplicates makes
the engine build the whole duplicated set and then sort it away. Aggregating first, or using `EXISTS`, never
creates the duplicates.

And the meta-rule: **do less work**. Filter earlier, return fewer columns, aggregate before joining, and do not
compute what nobody reads.

In [ ]:
correlated = """
SELECT p.product_id,
       p.name,
       (SELECT COUNT(*)      FROM order_items i WHERE i.product_id = p.product_id) AS lines,
       (SELECT SUM(quantity) FROM order_items i WHERE i.product_id = p.product_id) AS units
FROM products p
ORDER BY p.product_id
"""

joined = """
WITH sold AS (
    SELECT product_id, COUNT(*) AS lines, SUM(quantity) AS units
    FROM order_items
    GROUP BY product_id
)
SELECT p.product_id,
       p.name,
       COALESCE(s.lines, 0) AS lines,
       COALESCE(s.units, 0) AS units
FROM products p
LEFT JOIN sold s ON s.product_id = p.product_id
ORDER BY p.product_id
"""

a = benchmark("two correlated subqueries", correlated)
b = benchmark("aggregate once, then join", joined)
print(f"{a / b:.1f}x, and the results are identical:",
      q(correlated).fillna(0).values.tolist() == q(joined).values.tolist())

print("\nOR across two columns -- one index cannot serve both halves:")
run("CREATE INDEX IF NOT EXISTS idx_events_amount ON events(amount)")
print(" ", q("""EXPLAIN QUERY PLAN
    SELECT COUNT(*) FROM events WHERE customer_id = 17 OR amount > 495""")["detail"].tolist())
print("split into two indexed halves:")
print(" ", q("""EXPLAIN QUERY PLAN
    SELECT COUNT(*) FROM (
        SELECT event_id FROM events WHERE customer_id = 17
        UNION
        SELECT event_id FROM events WHERE amount > 495)""")["detail"].tolist())

**Step by step:**

1. The correlated version runs two subqueries for each of the 40 products — 80 passes over `order_items`.
2. The rewrite aggregates `order_items` **once** into `sold`, then joins. One pass, and `COALESCE` supplies the
   zeros for products that never sold, which the correlated `SUM` returned as `NULL`.
3. `.fillna(0).equals(...)` confirms the two produce the same numbers. Never accept a rewrite you have not
   checked against the original — most "optimizations" that break something break it quietly.
4. The `OR` plan says `MULTI-INDEX OR`: SQLite has already done the rewrite for you, running each half against
   its own index and merging. The manual `UNION` version produces almost the same plan — which is the point of
   section 2. Check before you rewrite.
5. Not every engine does this. MySQL's index-merge is narrower and older PostgreSQL versions often just scan, so
   the manual split is still worth knowing — and on a large table on disk it frequently wins. The real lesson is
   to measure the rewrite on data the size of the data you actually have.

## 21. Anti-Patterns

Things that are legal, common, and wrong.

| Anti-pattern | Why it hurts |
| --- | --- |
| `SELECT *` in saved code | breaks when a column is added, blocks covering indexes, drags data nobody reads |
| Function on an indexed column in `WHERE` | index unusable; rewrite as a range |
| Implicit type conversion | `WHERE id = '17'` on an integer column silently defeats the index |
| `OFFSET` for deep pagination | the engine still walks the skipped rows; use a keyset cursor |
| `DISTINCT` to hide duplicates | builds the duplicates, then sorts them away, and hides the real bug |
| `COUNT(*)` after a `LEFT JOIN` | counts the unmatched placeholder rows |
| Storing dates as `'DD/MM/YYYY'` | no correct sorting, no range filters, no date arithmetic |
| Comma-separated values in a column | cannot be indexed, joined, or counted correctly |
| One index per column, added reactively | eight indexes, no composite ones, every write eight times slower |
| Indexing a low-selectivity column | never used; pure write cost |
| `NOT IN (subquery)` over a nullable column | returns zero rows, silently |
| Building SQL by string concatenation | SQL injection, and no statement reuse |
| A transaction held open across a network call | locks held for seconds; everything queues behind it |
| No `ORDER BY` with `LIMIT` | results change between runs |

The two most expensive on a real system are usually the N+1 loop and the missing index on a foreign key. Both
are invisible in the code — nothing looks wrong — and both are found the same way: log the slow queries, take
the top one, and read its plan.

In [ ]:
run("DROP TABLE IF EXISTS typed")
run("""
CREATE TABLE typed (id INTEGER PRIMARY KEY, ref_id INTEGER, label TEXT);
INSERT INTO typed (id, ref_id, label)
WITH RECURSIVE s(n) AS (SELECT 1 UNION ALL SELECT n + 1 FROM s WHERE n < 50000)
SELECT n, n % 500, 'row-' || n FROM s;
""")
run("CREATE INDEX idx_typed_ref ON typed(ref_id)")

print("integer compared to integer:")
print("  ", q("EXPLAIN QUERY PLAN SELECT * FROM typed WHERE ref_id = 42")["detail"][0])
print("integer compared to text:")
print("  ", q("EXPLAIN QUERY PLAN SELECT * FROM typed WHERE CAST(ref_id AS TEXT) = '42'")["detail"][0])

print("\ndeep pagination:")
print(f"  OFFSET 40000: {timed('SELECT * FROM typed ORDER BY id LIMIT 20 OFFSET 40000'):6.2f} ms")
print(f"  keyset       : {timed('SELECT * FROM typed WHERE id > 40000 ORDER BY id LIMIT 20'):6.2f} ms")

def bulk_insert():
    t0 = time.perf_counter()
    run("""INSERT INTO typed (ref_id, label)
           WITH RECURSIVE s(n) AS (SELECT 1 UNION ALL SELECT n + 1 FROM s WHERE n < 40000)
           SELECT n % 500, 'bulk-' || n FROM s;""")
    return (time.perf_counter() - t0) * 1000

print("\nwhat every extra index costs you on writes:")
run("DROP INDEX IF EXISTS idx_typed_ref")
bare = bulk_insert()
run("""CREATE INDEX idx_typed_ref   ON typed(ref_id);
       CREATE INDEX idx_typed_label ON typed(label);
       CREATE INDEX idx_typed_both  ON typed(ref_id, label);
       CREATE INDEX idx_typed_id_ref ON typed(id, ref_id);""")
indexed = bulk_insert()
print(f"  40,000 rows, no indexes : {bare:7.1f} ms")
print(f"  40,000 rows, 4 indexes  : {indexed:7.1f} ms   ({indexed / bare:.1f}x slower)")

**Step by step:**

1. `WHERE ref_id = 42` uses the index. `WHERE CAST(ref_id AS TEXT) = '42'` cannot, because the function has
   moved the column out of reach — the same SARGability rule as section 7, arriving through a type mismatch
   rather than a deliberate function call.
2. In a real application that `CAST` is usually invisible: an ORM binds a string, or a column is `VARCHAR` on
   one side of a join and `INT` on the other.
3. `OFFSET 40000` makes the engine produce and discard 40,000 rows. The keyset version jumps straight there via
   the primary key and reads 20. The gap grows linearly with the offset, which is why page 500 of a listing is
   always the slow one.
4. Keyset pagination needs the last id from the previous page rather than a page number, so it changes your
   API. That is the trade, and it is the right one for anything with more than a few pages.
5. The last block is the cost nobody measures. The same 40,000-row insert takes several times longer with
   four indexes on the table, because every one of them has to be updated per row. An index that no query
   actually uses is pure overhead — audit `sqlite_master`, check the plans, and drop the ones nothing chose.

## 22. Dialect Differences at This Level

The further you go from `SELECT`, the more the databases diverge. What you have learned still applies; the
words move.

| Task | SQLite | PostgreSQL | MySQL 8+ |
| --- | --- | --- | --- |
| See the plan | `EXPLAIN QUERY PLAN` | `EXPLAIN (ANALYZE, BUFFERS)` | `EXPLAIN ANALYZE` / `EXPLAIN FORMAT=JSON` |
| Refresh statistics | `ANALYZE` | autovacuum, or `ANALYZE` | `ANALYZE TABLE` |
| Index extra columns | put them in the key | `INCLUDE (col)` | put them in the key |
| Partial index | `WHERE` on `CREATE INDEX` | `WHERE` on `CREATE INDEX` | not supported |
| Expression index | supported | supported | supported (8.0.13+) |
| Generate a series | recursive CTE | `generate_series(a, b)` | recursive CTE |
| Recursive CTE | `WITH RECURSIVE` | `WITH RECURSIVE` | `WITH RECURSIVE` |
| Upsert | `ON CONFLICT DO UPDATE` | `ON CONFLICT DO UPDATE` | `ON DUPLICATE KEY UPDATE` |
| Merge | not supported | `MERGE` (15+) | not supported |
| JSON access | `json_extract(d, '$.k')` | `d->>'k'` | `d->>'$.k'` |
| Materialized view | no | `CREATE MATERIALIZED VIEW` | no |
| Default isolation | serializable | read committed | repeatable read |
| Concurrent writers | one at a time | many | many |

The two that will actually bite you when you move off SQLite:

- **Concurrency.** SQLite serialises writers. PostgreSQL and MySQL do not, so the deadlocks, lock waits and
  isolation anomalies in section 13 stop being theory. Consistent lock ordering matters there.
- **Type strictness.** SQLite stores whatever you give it. PostgreSQL will reject it. Code that works here can
  fail there — which, for a learning database, is the safer direction to be wrong in.

In [ ]:
print("SQLite", q("SELECT sqlite_version() AS v")["v"][0])

# a partial index: index only the rows anybody queries
run("DROP INDEX IF EXISTS idx_orders_open")
run("CREATE INDEX idx_orders_open ON orders(order_date) WHERE status IN ('placed', 'shipped')")

print("\nquery matching the partial index's WHERE:")
print("  ", q("""EXPLAIN QUERY PLAN
    SELECT * FROM orders WHERE status IN ('placed', 'shipped') AND order_date >= '2024-06-01'
""")["detail"][0])
print("query that does not match it:")
print("  ", q("""EXPLAIN QUERY PLAN
    SELECT * FROM orders WHERE status = 'delivered' AND order_date >= '2024-06-01'
""")["detail"][0])

print("\nindex sizes on this database:")
print(q("""
SELECT name, tbl_name FROM sqlite_master WHERE type = 'index' AND sql IS NOT NULL
ORDER BY tbl_name, name
""").to_string(index=False))

**Step by step:**

1. A **partial index** covers only the rows matching its own `WHERE`. Open orders are a small fraction of the
   table, so the index is small and cheap to maintain.
2. A query whose conditions imply the index's `WHERE` can use it. The plan confirms a search.
3. A query for `'delivered'` cannot — those rows are not in the index at all — so it falls back to a scan. That
   is correct behaviour, not a failure.
4. This is the right tool for "status = pending" style queries on a big table where 99% of the rows are done and
   nobody looks at them. PostgreSQL supports it identically; MySQL does not have it.
5. The last query lists the indexes created through this notebook. `sql IS NOT NULL` filters out the internal
   ones SQLite makes for primary keys. Auditing what exists is the first step before adding anything.

## 23. Classic Interview Problems

Six problems that come up constantly. Each has a one-line idea behind it.

1. **Second-highest value.** `DENSE_RANK() OVER (ORDER BY x DESC) = 2`. Handles ties correctly, unlike
   `ORDER BY x DESC LIMIT 1 OFFSET 1`, and returns nothing rather than the wrong row when there is no second
   value.
2. **Top N per group.** `ROW_NUMBER()` partitioned by the group, filtered outside the CTE. You know this one.
3. **Running balance.** `SUM(amount) OVER (PARTITION BY account ORDER BY d ROWS UNBOUNDED PRECEDING)`.
4. **Median.** No `MEDIAN` in standard SQL. Number the rows ascending and descending, and take the rows where
   the two numbers are within one of each other — that middle works for odd and even counts alike.
5. **Funnel / conversion.** Conditional aggregation over per-customer flags: did they view, did they cart, did
   they check out. `COUNT(DISTINCT CASE WHEN ... END)` at each step.
6. **Consecutive runs.** Gaps and islands, section 12.

What an interviewer is usually watching for: do you state the grain, do you handle ties, do you handle `NULL`,
and do you write it as CTEs somebody else can read.

In [2]:
print("second-highest salary, ties handled:")
print(q("""
WITH ranked AS (
    SELECT name, role, salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS dr
    FROM employees
)
SELECT name, role, salary FROM ranked WHERE dr = 2
""").to_string(index=False))

print("\nmedian order value, without a MEDIAN function:")
print(q("""
WITH order_totals AS (
    SELECT o.order_id, SUM(i.quantity * i.unit_price * (1 - i.discount)) AS total
    FROM orders o JOIN order_items i ON i.order_id = o.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY o.order_id
),
numbered AS (
    SELECT total,
           ROW_NUMBER() OVER (ORDER BY total)      AS asc_n,
           ROW_NUMBER() OVER (ORDER BY total DESC) AS desc_n
    FROM order_totals
)
SELECT ROUND(AVG(total), 2) AS median_order_value
FROM numbered
WHERE asc_n BETWEEN desc_n - 1 AND desc_n + 1
""").to_string(index=False))

print("\nconversion funnel, one visit = one customer on one day:")
q("""
WITH per_visit AS (
    SELECT customer_id,
           event_date,
           MAX(CASE WHEN event_type = 'view'     THEN 1 ELSE 0 END) AS viewed,
           MAX(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) AS carted,
           MAX(CASE WHEN event_type = 'checkout' THEN 1 ELSE 0 END) AS checked_out
    FROM events
    WHERE event_date >= '2024-01-01'
    GROUP BY customer_id, event_date
)
SELECT COUNT(*)                                          AS visits,
       SUM(viewed)                                       AS viewed,
       SUM(carted)                                       AS carted,
       SUM(checked_out)                                  AS checked_out,
       ROUND(100.0 * SUM(carted)      / SUM(viewed), 1)  AS view_to_cart_pct,
       ROUND(100.0 * SUM(checked_out) / SUM(carted),  1) AS cart_to_checkout_pct
FROM per_visit
""")

second-highest salary, ties handled:


NameError: name 'q' is not defined

**Step by step:**

1. `DENSE_RANK() OVER (ORDER BY salary DESC) = 2` gives the second-highest **salary**, and everybody on it. Two
   people on the same salary both appear, which is almost always the intended answer.
2. The median trick numbers each order both ways. In the middle of the list the ascending and descending
   positions meet; `BETWEEN desc_n - 1 AND desc_n + 1` catches the one middle row when the count is odd and the
   two middle rows when it is even, and `AVG` then does the right thing in both cases.
3. The funnel first collapses to one row per **visit** — a customer on a day — with three yes/no flags.
   `MAX(CASE WHEN ...)` is the idiom for "did this ever happen", because `MAX` of 0s and 1s is 1 if any row
   matched.
4. The grain choice is the whole exercise. Per customer, everybody eventually does everything and the funnel
   reads 100% at every step; per visit, you see the real drop-off. Counting raw events would be worse still —
   one very active customer would dominate.
5. Notice all three are the same three moves: get to the right grain in a CTE, apply a window function or a
   conditional aggregate, then filter or divide outside. Nearly every hard SQL question is that shape.

## Cheat Sheet

| Task | SQL |
| --- | --- |
| See the plan | `EXPLAIN QUERY PLAN SELECT ...` |
| Refresh statistics | `ANALYZE` |
| Plan line to fear | `SCAN t` on a big table |
| Plan line to want | `SEARCH t USING COVERING INDEX i` |
| Sort happened at run time | `USE TEMP B-TREE FOR ORDER BY` |
| Create an index | `CREATE INDEX idx ON t(col)` |
| Composite index | `CREATE INDEX idx ON t(equality_col, range_col)` |
| Covering index | add the selected columns to the end of the key |
| Partial index | `CREATE INDEX idx ON t(col) WHERE status = 'open'` |
| Expression index | `CREATE INDEX idx ON t(strftime('%Y', d))` |
| List indexes | `SELECT name, sql FROM sqlite_master WHERE type = 'index'` |
| SARGable date filter | `d >= '2024-01-01' AND d < '2025-01-01'` |
| Not SARGable | `strftime('%Y', d) = '2024'`, `LOWER(x) = ...`, `LIKE '%x%'` |
| Time a query fairly | warm up, then take the median of several runs |
| Measure server work only | `SELECT COUNT(*) FROM ( ... )` |
| Hierarchy | `WITH RECURSIVE t AS (anchor UNION ALL step JOIN t)` |
| Generate dates | `SELECT date(d, '+1 day') FROM spine WHERE d < ...` |
| Fill missing periods | spine `LEFT JOIN` data, then `COALESCE(x, 0)` |
| Running total, no tie surprises | `SUM(x) OVER (ORDER BY d ROWS UNBOUNDED PRECEDING)` |
| Reuse a window | `... OVER w ... WINDOW w AS (PARTITION BY g ORDER BY d)` |
| Islands | `date(d, '-' \|\| ROW_NUMBER() OVER (ORDER BY d) \|\| ' days')`, then group |
| Sessionize | `LAG` the timestamp, flag big gaps, running `SUM` of the flags |
| Transaction | `BEGIN; ...; COMMIT;` / `ROLLBACK;` |
| Better concurrency in SQLite | `PRAGMA journal_mode = WAL` |
| Cascade a delete | `REFERENCES parent(id) ON DELETE CASCADE` |
| Derived column that cannot drift | `GENERATED ALWAYS AS (expr) STORED` |
| Unique combination | `UNIQUE (order_id, product_id)` |
| Idempotent daily load | `DELETE FROM f WHERE date_key = ?` then `INSERT`, in one transaction |
| Idempotent row load | `INSERT ... ON CONFLICT (key) DO UPDATE SET ...` |
| JSON field | `json_extract(doc, '$.a.b')` |
| JSON array to rows | `FROM t, json_each(t.doc, '$.tags')` |
| Index a JSON field | generated `STORED` column over `json_extract`, then index it |
| Second highest | `DENSE_RANK() OVER (ORDER BY x DESC) = 2` |
| Median | number ascending and descending, take where they meet |
| Funnel | `MAX(CASE WHEN step THEN 1 ELSE 0 END)` per entity, then sum |
| Keyset pagination | `WHERE id > :last_id ORDER BY id LIMIT 20` |

## Suggested Learning Path

1. Run section 1 and confirm `events` has 200,000 rows. Nothing after this works without it.
2. Read sections 2 and 3 and get comfortable putting `EXPLAIN QUERY PLAN` in front of things. Do it to queries
   you wrote in the earlier levels.
3. Sections 4 to 7 are indexing. Do exercises 4 to 7 immediately after — indexing is learned by watching plans
   change, not by reading.
4. Section 8 is short and worth taking seriously. Every performance claim you make from here should survive it.
5. Sections 9 and 10 are recursion. The date spine in section 10 is the one you will use most often.
6. Sections 11 and 12 finish window functions. Write the islands query from memory until you can.
7. Sections 13 and 14 are the correctness half: transactions and constraints. Read 13 twice if you have never
   thought about isolation.
8. Sections 15 to 18 are design and loading. If you build data pipelines, these four matter more than the
   indexing sections.
9. Section 19 only if you have JSON in your life. It is self-contained.
10. Sections 20 and 21 are the practical summary — rewrites that work, patterns that do not.
11. Section 23 last, then the exercises end to end. The two mini projects are the level in miniature.

## Where to Go Next

- Get a real PostgreSQL running, load a few million rows, and repeat sections 4 to 8 there. `EXPLAIN (ANALYZE,
  BUFFERS)` showing estimated versus actual rows is the single most useful tool in this entire subject, and
  SQLite has no equivalent.
- Read *SQL Performance Explained* by Markus Winand, or his free site [use-the-index-luke.com](https://use-the-index-luke.com).
  It is about indexes and nothing else, and it is the best thing written on them.
- Take the slowest query at your work, read its plan, and fix it. Write down the before and after timings and
  the plan change. That single exercise teaches more than the rest of this list.
- For the modelling half, Kimball's dimensional modelling material is still the reference for star schemas and
  slowly changing dimensions.

## Practice Next

Open `sql-advanced-exercises.ipynb`. The performance exercises ask you to read and change query plans rather
than to memorise rules, and the two mini projects — optimising a slow report end to end, and building and
loading a star schema — are the ones worth keeping.